# UCS420: Cognitive Computing
## Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0)

# Q1: Build Your Personalized Knowledge Base

Take your college roll number. Extract its digits. Build a **Pandas DataFrame with exactly 6 FAQ entries**:

- **4 fixed entries** given in the table below.
- **2 additional entries** constructed from your own roll number digits as follows:

### Instructions

- Take the **LAST TWO DIGITS** of your roll number.
- For each digit `d`, compute the category using:

```text
category = ["billing", "account", "general"][d % 3]

In [ ]:
import pandas as pd

ROLL_NUMBER = "1024170032"

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

digit_category = {
    0: "billing",
    1: "account",
    2: "general"
}

last_two_digits = [int(d) for d in ROLL_NUMBER[-2:]]

personalized_entries = [
    {
        "question": "how can i update my registered mobile number",
        "answer": "Go to Account Settings and update your registered mobile number.",
        "keywords": "mobile number update",
        "category": digit_category[last_two_digits[0] % 3]
    },
    {
        "question": "where can i check the help information",
        "answer": "You can check the help section for general information.",
        "keywords": "help information support",
        "category": digit_category[last_two_digits[1] % 3]
    }
]

faq_entries = fixed_entries + personalized_entries
df = pd.DataFrame(faq_entries)

print("Final 6-row FAQ DataFrame:")
display(df)

## Q2. Generate and Score a Hypothesis  Implement a scoring function that takes a query string and returns all matching entries ranked by confidence 

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        text = (
            row["question"] + " " +
            row["answer"] + " " +
            row["keywords"]
        ).lower()

        score = sum(1 for word in query_words if word in text.split())

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    return pd.DataFrame(results).sort_values(
        by="confidence", ascending=False
    ).reset_index(drop=True)

query = input("Enter your query: ")
matches = score_query(query, df)

if matches.empty:
    print("No matching FAQ entries found.")
else:
    display(matches)

## Q3. Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result. 

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]

personalized_category = personalized_entries[0]["category"]

print("Category selected from personalized entry:", personalized_category)
display(same_category(personalized_category, df))

## Q4. Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv. 

In [ ]:
entry_index = 0

print("Selected FAQ entry:")
display(df.loc[[entry_index]])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    current_keywords = df.at[entry_index, "keywords"].split()
    if new_keyword not in current_keywords:
        current_keywords.append(new_keyword)
        df.at[entry_index, "keywords"] = " ".join(current_keywords)
        print("Keyword added successfully.")
    else:
        print("Keyword already exists.")
else:
    print("No keyword entered.")

output_file = f"{ROLL_NUMBER}_faq_data.csv"
df.to_csv(output_file, index=False)

print(f"Updated DataFrame saved as: {output_file}")
display(df)

## Q5. Count FAQ Entries per Category Using `groupby`

In [ ]:
category_counts = df.groupby("category").size().reset_index(name="count")

print("Number of FAQ entries per category:")
display(category_counts)

## Q6.  Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't. 

In [ ]:
def score_query_with_ties(query, df):
    results = []

    for index, row in df.iterrows():
        query_words = set(query.lower().split())
        text = (
            row["question"] + " " +
            row["answer"] + " " +
            row["keywords"]
        ).lower()

        score = sum(1 for word in query_words if word in text.split())

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    if not results:
        print("No matching FAQ entries found.")
        return pd.DataFrame()

    result_df = pd.DataFrame(results).sort_values(
        by="confidence", ascending=False
    ).reset_index(drop=True)

    highest_score = result_df["confidence"].max()
    top_matches = result_df[result_df["confidence"] == highest_score]

    if len(top_matches) > 1:
        print("Tie detected! All highest-confidence matches are shown:")
        display(top_matches)
    else:
        print("No tie. Highest-confidence match:")
        display(top_matches)

    return result_df

print("----- Tie demonstration -----")
tie_query = "fee"
score_query_with_ties(tie_query, df)

print("----- Non-tie demonstration -----")
non_tie_query = "password"
score_query_with_ties(non_tie_query, df)